## 解説


有限個の訓練データを用いて回帰を行うために以下の図の？で示した領域ように外挿となる（はずの）説明変数領域は存します。

![extrapolation](image_keep/fig_extrapolation.PNG)

回帰によりある説明変数での目的変数値が予測可能なことは分かりましたが、
同時に値の不確かさ（例えば標準偏差）は評価できないでしょうか。
観測値にノイズが含まれるとして定式化を行うのがその一つの手法です。

### ガウス過程
予測モデルとして
$$
f(x) = \sum_p c_p K(x,x_p) 
$$
$K(x,x')$としてガウシアン
$$
K(x,x') = \exp\left(-\gamma |\!|x-x'|\!|_2^2) \right)  
$$
を考えた予測モデルをガウス過程（ガウシアンプロセス）と呼びます。
ここに$x_p$は訓練データの説明変数で、$c_p$は回帰により求める係数です。

ガウス過程は
訓練データ点$x$に$K(x,x')$で表現される確率分布を置いていくという考え方で、
以下の図のように訓練データ点では誤差を小さく、訓練データが無い点では誤差を大きくする予測ができます。
ここで緑色の破線は平均値＋標準偏差と平均値ー標準偏差を意味します。

![GaussianProcess put probability](image_keep/fig_gaussianprocess1.PNG)

### ベイズ最適化

ベイズ最適化はこのガウス過程が予測標準偏差が求まることを用いて獲得関数を通して最適な説明変数探索を行う手法です。

1. 既知の候補点($x_i,y_{\textrm{expr},i}$)が存在する。($i=1,...,n$)
2. ガウス過程により未探索点jの$y_{\textrm{mean},j}$とその標準偏差$\sigma_j$を予測する。
3. $y_{\textrm{mean},j}$と$\sigma_j$を利用した獲得関数により、次候補点のscoreを評価する。
最も高い獲得関数scoreの点を評価し$y_{\textrm{expr},k}$を得て、既知の候補点セットに入れる。
4. 最初に戻る。

という作業を逐次的に行います。
この反復が難しい場合は初回の獲得関数scoreのリストを与え、その中で実際に値を評価するだけでも十分な場合があります。




#### 獲得関数

獲得関数には

1. Probability of improvement (PI)
2. Expected improvement (EI)
3. Upper confidence bound (UCB)
4. Thompson sampling (TS)

…
など色々あります。


例えばUCB (upper confidence bound)と呼ばれる獲得関数を用いる場合は以下の図の①から⑥のような探索を行う事になります。

![GaussianProcess regression](image_keep/fig_gaussianprocess2.PNG)

ベイズ最適化は予測平均値だけでなく大きな標準偏差で表現される未評価点を考慮して獲得関数を定量的に評価する手法です。
訓練データの予測平均値による最適値選択（＝活用）と大きな標準偏差で表現される未評価点選択（＝探索）を行う手法であると言われます。

ベイズ最適化は
微分が評価できない関数に対して最適解を求めることが可能にする手法です。
標準偏差の大きさから
未探索点の評価がある程度できるので全空間の探索が可能です。


### 獲得関数

代表的な獲得関数, aquisition function ($a$)を以下に書きます。

$\phi(x)$　を正規分布関数、 
$\Phi(x) = \int^x_{-\infty} dx'\phi(x')$　をその積分（累積分布関数）とします。

繰り返しの回数を$t$とします。

#### Probability of Improvement (PI)

$$z = (y_\textrm{mean}-f^+- \xi)/\sigma $$

として

$$a_{\textrm{PI}} = \Phi(z)$$

$f^+$ は候補点の中での最大値です。
$\xi$ はある定数です。

#### Expected Improvement (EI)


$$ z = (y_\textrm{mean} - f^+ - \xi)/ \sigma $$
$$a_{\textrm{EI}} = (y_\textrm{mean} - f^+ - \xi) \Phi(z)+ \sigma \phi(z)$$

これには変種があります。

#### Upper Confidence Bound (UCB)

$$a_{\textrm{UCB}} = y_\textrm{mean} + k_t  \sigma$$ 

ここで $k_t$は $\sqrt{v t}$ とするか定数とします。


#### Thompson sampling (TS)

確率過程を用いて獲得関数とします。
ガウス過程はサンプル点の平均値だけでなく、共分散行列を与えます。


$$
f(x) = \frac{1}{(2\pi)^k \det(\Sigma)} \exp\left( -
\frac{1}{2} (x-\mu)^T \Sigma^{-1} (x-\mu)
\right)
$$

ここで $\mu$は$x$の平均値、$\Sigma$は共分散行列で（予測点の数、予測点の数）のサイズを持ちます。
また
$\mu$ と $x$はベクトルで$k$は$x$のサイズです。
高次元のガウス分布が定義できるのでそこから一点サンプルを選びます。
これは、例えば、scipyのmultivariate_normal.rvs()で行えます。


ref.

multivariate_normal

https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.multivariate_normal.html

## 補足

ベイズ最適化は元々はベイズの理論を用いた回帰モデル、例えばガウス過程回帰、を用いる定式化だった。しかし、最近は予測値の平均値と分散を出力できる回帰モデル、例えばアンサンブル決定木回帰、に対しても用いられる用語です。

## 注意

簡単な説明変数を用いた回帰モデル、もしくは単純な回帰モデル（高速な学習と高速な予測ができ＝容易にパラメタ変化に対する目的変数値が評価できる「代理モデル」と呼ばれる）を用いるので、必ずしも観測データに対する最適な観測値を与える説明変数が求まるわけではありません。代理モデルの最適値を与える説明変数が求まります。



#### 参考文献

1. Bayesian optimzation
Jasper Snoek, Hugo Larochelle and Ryan P. Adams, 
"PRACTICAL  BAYESIAN  OPTIMIZATION  OF  MACHINE  LEARNING
ALGORITHMS"
https://arxiv.org/pdf/1206.2944.pdf

2. https://qiita.com/masasora/items/cc2f10cb79f8c0a6bbaa


